In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from gbm_utils import simulate_gbm

In [3]:
#plot 한글 적용
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [22]:
#Black-Scholes 공식
def black_scholes_analytical(S_0, K, T, r, sigma, type='call'):
    d1 = (np.log(S_0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))   #Black-Scholes 공식
    d2 = (np.log(S_0 / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))

    # norm.cdf() : 표준정규분포의 누적분포함수
    # np.exp() : 미래의 돈을 현재 가치로 할인
    if type == 'call':
        val = (S_0 * norm.cdf(d1, 0, 1) - K * np.exp(-r * T) * norm.cdf(d2, 0, 1))
    elif type == 'put':
        val = (K * np.exp(-r * T) * norm.cdf(-d2, 0, 1) - S_0 * norm.cdf(-d1, 0, 1))
    else:
        raise ValueError('Wrong input for type!')
        
    return val

In [23]:
#Monte Carlo 시뮬레이션
def european_option_simulation(S_0, K, T, r, sigma, n_sims, type='call', random_seed=42):
    np.random.seed(random_seed)
    rv = np.random.normal(0, 1, size=n_sims)
    S_T = S_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * rv)

    if type == 'call':
        payoff = np.maximum(0, S_T - K)
    elif type == 'put':
        payoff = np.maximum(0, K - S_T)
    else: 
        raise ValueError('Wrong input for type!')
        
    premium = np.mean(payoff) * np.exp(-r * T)
    return premium

#### 가격 평가 매개변수

In [6]:
S_0 = 100                        #최초주가
K = 100                          #스트라이트 가격(만기 시 매수/매도 가격)
r = 0.05                         #연간 무위험 금리
sigma = 0.50                     #기초 주식 변동성
T = 1                            #만기 기간(year)
N = 252                          #시뮬레이션 시간 증분
dt = T / N                       #시간 간격 설정
N_SIMS = 1000000                 #시물레이션 수
discount_factor = np.exp(-r * T) #할인 요인

In [12]:
#콜옵션 평가
black_scholes_analytical(S_0=S_0, K=K, T=T, r=r, sigma=sigma, type='call')

np.float64(21.79260421286685)

#### 시뮬레이션 실행

In [13]:
gbm_sims = simulate_gbm(s_0=S_0, mu=r, sigma=sigma, n_sims=N_SIMS, T=T, N=N)

In [18]:
#시뮬레이션 결과를 df로 변환
gbm_sims_df = pd.DataFrame(np.transpose(gbm_sims))

In [20]:
#옵션 프리미엄 계산
premium = discount_factor * np.mean(np.maximum(0, gbm_sims[:, -1] - K))
premium

np.float64(21.73193752739459)

In [24]:
european_option_simulation(S_0, K, T, r, sigma, N_SIMS, type='put')

np.float64(16.948225203893127)